In [11]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('../..')
import data_preprocessing.coil20 as coil20
from importlib import reload
reload(coil20)

<module 'data_preprocessing.coil20' from '/Users/apple/Library/Mobile Documents/3L68KQB4HG~com~readdle~CommonDocuments/Documents/本学期资料/MSBD5002Project/implementation/experiment/1_regularization_term/../../data_preprocessing/coil20.py'>

In [20]:
mat, label = coil20.read_data()
mat = torch.tensor(mat, dtype=torch.float32)
A = mat @ mat.T
N = A.shape[0]

k = 8
H = torch.randn(N, k)

In [27]:
def loss_without_regularization(A, H):
    '''
    A: a symmetric matrix of size n x n
    H: a matrix of size n x k
    '''
    assert A.T.equal(A), "A is not symmetric"
    
    return torch.norm(A - H @ H.T) ** 2

# def loss_with_regularization(A, H, lambda_):
#     '''
#     A: a symmetric matrix of size n x n
#     H: a matrix of size n x k
#     lambda_: a scalar
#     '''
#     assert A.T.equal(A), "A is not symmetric"
    
#     loss = torch.norm(A - H @ H.T) ** 2
    
#     regularization_loss = 0
#     for i in range(A.shape[0]):
#         for j in range(A.shape[1]):
#             regularization_loss += A[i, j] * torch.norm(H[i] - H[j]) ** 2
#     regularization_loss *= lambda_
    
#     return loss + regularization_loss

def loss_with_regularization_optimized(A, H, lambda_):
    '''
    A: a symmetric matrix of size n x n
    H: a matrix of size n x k
    lambda_: a scalar
    '''
    assert A.T.equal(A), "A is not symmetric"
    
    loss = torch.norm(A - H @ H.T) ** 2
    
    # Vectorized computation of regularization loss
    diff = H[:, None, :] - H[None, :, :]
    regularization_loss = (A.unsqueeze(-1) * diff**2).sum()
    regularization_loss *= lambda_
    
    return loss + regularization_loss

In [28]:
loss_with_regularization(A, H, 0.1)


tensor(4.3626e+10)

In [ ]:
def gd(data, k, lamda, max_iter=100, tol=1e-4, print_every=100):